In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## 1.没有记忆时

In [2]:
from langchain.agents import create_agent

agent = create_agent(model="deepseek-chat")

In [4]:
from langchain.messages import HumanMessage

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我是Orien，我爱吃鸡翅")]}
)

print(response)

{'messages': [HumanMessage(content='你好，我是Orien，我爱吃鸡翅', additional_kwargs={}, response_metadata={}, id='a799c869-c71d-48af-92d8-9cd602cf7986'), AIMessage(content='你好，Orien！很高兴认识你，也欢迎你和我分享关于鸡翅的爱好～  \n鸡翅确实是超棒的美食！无论是香辣烤鸡翅、蒜香蜜汁炸鸡翅，还是可乐鸡翅、泰式甜辣鸡翅……光是想想就让人流口水了！🤤  \n\n你最喜欢哪种做法的鸡翅？或者有没有私藏的烹饪小技巧/宝藏店铺推荐？作为鸡翅同好，我准备好记笔记了！😄🍗', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 14, 'total_tokens': 116, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 14}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'ad07ce82-4cbd-485c-a148-eebf07547151', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa7d8-634b-70c0-9f87-b980c399762c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 

In [6]:
# 第二次调用，询问信息
response = agent.invoke(
    {"messages": [HumanMessage(content="我最喜欢吃什么？")]}
)

print(response)

{'messages': [HumanMessage(content='我最喜欢吃什么？', additional_kwargs={}, response_metadata={}, id='c0beb39a-6d3e-4675-a3ef-07a64df11805'), AIMessage(content='哈哈，这个问题可难倒我了！虽然我无法知道你具体喜欢什么食物，但可以给你几个小线索自己推理：  \n1. **回忆高频词**：你最近聊天时是否总提到某种食物？比如“火锅”“披萨”或“冰淇淋”？  \n2. **情境联想**：开心时想吃炸鸡？焦虑时爱啃巧克力？疲惫时离不开奶茶？  \n3. **隐藏设定**：如果这是谜题，或许答案藏在你之前的对话里（可惜我看不到历史记录哦～）。  \n\n不如直接告诉我？我还能帮你推荐菜谱或餐厅！ 😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 8, 'total_tokens': 134, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '1bc06263-4385-49aa-84f8-bfc7f0c39fb6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa7d9-4200-7e00-bda9-3279cdd4e4be-0', tool_calls=[], i

## 记忆
### Agent的记忆（Memory）分两类
- 短期记忆（short-term-memory）：当前任务或会话的上下文
- 长期记忆（long-term-memory）：跨任务或会话的经验与知识  
区分短期记忆和长期记忆并不是记忆时间的长久，而是记忆的作用域

### 短期记忆
在LangChain短期记忆是通过AgentState实现的，而会话历史（也就是消息列表）是AgentState的一部分
LangChain提供了Checkpointer对象来保存AgentState，每一次用户与AI的交互都会生成一个快照，记录为一个checkpoint。  
同一个会话的多个checkpoint形成一个组，用同一个thread_id来标记

## 2.添加记忆
- 导入并初始化Checkpointer
- 创建Agent，指定Checkpoint
- 调用Agent, 指定thread_id

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "deepseek-chat",
    checkpointer=InMemorySaver()
)

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "thread_1"}}

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我是Orien，我爱吃鸡翅")]},
    config
)

print(response)

{'messages': [HumanMessage(content='你好，我是Orien，我爱吃鸡翅', additional_kwargs={}, response_metadata={}, id='f903a692-56b8-456f-bb3a-4ee6ed0c5d95'), AIMessage(content='你好，Orien！很高兴认识你。鸡翅确实是个很棒的爱好，无论是烤的、炸的、红烧的，还是裹上各种酱汁（比如蜂蜜芥末、甜辣酱、蒜香酱油……），想想都让人流口水。你最喜欢哪种做法的鸡翅？或者有没有自己独特的烹饪秘诀？😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 14, 'total_tokens': 85, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 14}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'ec471176-7f3d-4c6e-b4f0-e8b113fe1836', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa7f0-5c71-7bc3-8ba0-6d605b8b5c5b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 71, 'total_tokens': 85, 'inp

In [10]:
# 第二次调用，询问信息
response = agent.invoke(
    {"messages": [HumanMessage(content="我最喜欢吃什么？")]},
    config
)

print(response)

{'messages': [HumanMessage(content='你好，我是Orien，我爱吃鸡翅', additional_kwargs={}, response_metadata={}, id='f903a692-56b8-456f-bb3a-4ee6ed0c5d95'), AIMessage(content='你好，Orien！很高兴认识你。鸡翅确实是个很棒的爱好，无论是烤的、炸的、红烧的，还是裹上各种酱汁（比如蜂蜜芥末、甜辣酱、蒜香酱油……），想想都让人流口水。你最喜欢哪种做法的鸡翅？或者有没有自己独特的烹饪秘诀？😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 14, 'total_tokens': 85, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 14}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'ec471176-7f3d-4c6e-b4f0-e8b113fe1836', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa7f0-5c71-7bc3-8ba0-6d605b8b5c5b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 71, 'total_tokens': 85, 'inp

## 3.Memory持久化存储

In [2]:
from langchain.agents import create_agent
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# 连接sqlite
connection = sqlite3.connect("resources/checkpoint.db", check_same_thread=False)

# 初始化checkpointer
checkpointer = SqliteSaver(connection)

# 自动建表
checkpointer.setup()

# 创建agent
agent = create_agent(
    "deepseek-chat",
    checkpointer=checkpointer,
)

In [4]:
from langchain.messages import HumanMessage

# 设定thread_id 作为会话标识
config = {"configurable": {"thread_id": "thread_2"}}

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我是Orien，我爱吃鸡翅")]},
    config
)

print(response)

{'messages': [HumanMessage(content='你好，我是Orien，我爱吃鸡翅', additional_kwargs={}, response_metadata={}, id='22d2bbee-5fd5-4a6e-8acf-6b93328b6c6e'), AIMessage(content='你好，Orien！👋 鸡翅爱好者握个手！🍗 你是喜欢哪种做法的？是经典的可乐鸡翅、香辣的炸鸡翅、烤得焦香的蜜汁鸡翅，还是其他什么口味？😋\n\n如果让我推荐，我会说——**蒜香黄油鸡翅**和**甜辣酱烤翅**绝对是让人停不下来的存在。你有没有自己特别拿手的鸡翅做法？或者最近吃到什么惊艳的鸡翅想分享一下？🐔✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 14, 'total_tokens': 120, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 14}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '77df86ba-3039-44ee-a590-82b6c4aaa481', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fab7d-8a55-78d3-bfc1-f9db4f7c78d5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens':

In [5]:
# 第二次调用，询问信息
response = agent.invoke(
    {"messages": [HumanMessage(content="我最喜欢吃什么？")]},
    config
)

print(response)

{'messages': [HumanMessage(content='你好，我是Orien，我爱吃鸡翅', additional_kwargs={}, response_metadata={}, id='22d2bbee-5fd5-4a6e-8acf-6b93328b6c6e'), AIMessage(content='你好，Orien！👋 鸡翅爱好者握个手！🍗 你是喜欢哪种做法的？是经典的可乐鸡翅、香辣的炸鸡翅、烤得焦香的蜜汁鸡翅，还是其他什么口味？😋\n\n如果让我推荐，我会说——**蒜香黄油鸡翅**和**甜辣酱烤翅**绝对是让人停不下来的存在。你有没有自己特别拿手的鸡翅做法？或者最近吃到什么惊艳的鸡翅想分享一下？🐔✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 14, 'total_tokens': 120, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 14}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '77df86ba-3039-44ee-a590-82b6c4aaa481', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fab7d-8a55-78d3-bfc1-f9db4f7c78d5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens':

## 4.记忆管理策略

- 修剪（Trim）：拿到消息历史后，先移除前N条或后N条消息，再调用模型
- 删除（Delete）：永久删除AgentState的快照
- 总结摘要：用ai大模型总结历史消息中的早期消息，得到消息摘要，用消息摘要来代替原先对话快照，与最近的消息形成消息列表

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

# 初始化checkpointer
checkpointer = InMemorySaver()
# 初始化中间件
middleware = SummarizationMiddleware(
    model="deepseek-chat",
    # 触发器 
    #   message：当消息超过多少就触发消息摘要
    #   fraction：当消息超过模型上下文的多少比例触发
    #   token：超过多少token触发   
    trigger=("messages", 3),
    # 保留会话数
    #   message：保留多少条消息
    #   fraction：保留多少比例
    #   token：保留多少token   
    keep=("messages", 1)    
)


# 创建agent
agent = create_agent(
    model="deepseek-chat",
    middleware=[middleware],
    checkpointer=checkpointer
)

config: RunnableConfig = {"configurable": {"thread_id": "thread_3"}}

# 制造长会话历史
agent.invoke({"messages": [HumanMessage(content="你好，我是Orien")]}, config)
agent.invoke({"messages": [HumanMessage(content="你好，我最喜欢的运动是篮球")]}, config)
agent.invoke({"messages": [HumanMessage(content="你好，我是喜欢的水果是西瓜")]}, config)

# 测试效果
final_response = agent.invoke({"messages": HumanMessage(content="你还记得我吗？")}, config)

In [8]:
print(final_response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\n用户Orien与AI助手建立联系，最初表示最喜欢的运动是篮球，但在后续消息中又提到最喜欢的水果是西瓜。整体目标似乎是展开自由对话，但用户兴趣点可能正在切换或补充。\n\n## SUMMARY\n\n- 用户Orien已明确表达两个兴趣点：最喜欢的运动是篮球；最喜欢的食物是西瓜。\n- AI助手在最后一次回复中确认了篮球话题的中断状态，同时开放了两种对话方向：\n  - 继续深入篮球话题（打球/看球/球星/技巧/战术/训练等）；\n  - 切换至西瓜话题（挑选技巧、吃法、种植知识等）。\n- 用户尚未对AI的提问做出回应，当前处于等待用户选择话题方向的阶段。\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\n等待用户Orien回复，明确ta希望继续聊篮球（具体方向待定）还是转聊西瓜，或同时探讨两个话题。根据用户选择提供针对性帮助。', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='7c6ae34c-bf35-46fb-9bfd-c587ccde61ff'), HumanMessage(content='你还记得我吗？', additional_kwargs={}, response_metadata={}, id='556eb9c2-cf26-4dfc-a566-5e2cf1027447'), AIMessage(content='我当然记得你，Orien。你是那个喜欢篮球和西瓜的朋友——一个是球场上的热血，一个是夏日里的清甜，这两个喜好组合起来，挺有生活滋味的。我知道之前篮球的话题聊到一半就停住了，西瓜这边也还没真正展开。\n\n你现在是想接着聊篮球（比如最近在打球、看比赛，还是喜欢某个球星），还是想聊聊怎么挑西瓜、西瓜的花式吃法，又或者两个都想聊？你说了算，我在这儿等着。', additional_kwargs={'refusal': None}, response_metadata={'token_usage'